# QData Free Source Factor API Arithmetic

This notebook is a concise walkthrough of deterministic adjusted reference arithmetic on the QData mock backend. It does not require Docker, paid data, pandas, or external network access.

The demo treats `momentum_20d` as an after-close signal on `2024-01-02`, ranks a synthetic signal-date-screened universe, and compares the next session's forward-adjusted open reference with its adjusted close mark. Next-session tradability is not verified. This is not an execution or backtest, executable-price claim, or real-market evidence.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "qdata").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from qdata import Client
from examples.factor_api_arithmetic_demo import run_demo, format_report

## 1. Build a signal-date-screened universe

The mock API applies suspension, ST, delisting-period, new-listing, and listing-day filters as of `2024-01-02`. This does not establish eligibility or tradability on the next session.

In [2]:
client = Client(default_format="records")
signal_universe = client.get_tradable_universe(
    asof_date="2024-01-02",
    universe="hs300",
    min_list_days=120,
)
signal_universe

[{'symbol': '600519.SH',
  'security_id': 1000001,
  'asof_date': '2024-01-02',
  'can_buy': True,
  'can_sell': True,
  'list_days': 8163,
  'is_st': False,
  'is_suspended': False,
  'is_new_listing': False,
  'is_delisting_period': False},
 {'symbol': '000001.SZ',
  'security_id': 1000002,
  'asof_date': '2024-01-02',
  'can_buy': True,
  'can_sell': True,
  'list_days': 11962,
  'is_st': False,
  'is_suspended': False,
  'is_new_listing': False,
  'is_delisting_period': False}]

## 2. Pull current fixture factor values

The public factor API supports strict latest/as-of/vintage selectors: `start_date` and `end_date` filter the factor's economic date, while a timezone-aware `asof_time` is the historical knowledge cutoff. The SQL backend admits only successful, finished, batch-bound, non-recalled dataset versions and fails closed on distinct payloads at one identity/version/calculation time; this notebook uses a synthetic mock fixture, so it checks API shape and deterministic timing only and is not PIT evidence.

In [3]:
symbols = [row["symbol"] for row in signal_universe]
client.get_factor(
    factors=["momentum_20d", "roe_ttm"],
    symbols=symbols,
    start_date="2024-01-02",
    end_date="2024-01-02",
    format="wide",
)

[{'symbol': '600519.SH',
  'security_id': 1000001,
  'trade_date': '2024-01-02',
  'momentum_20d': 0.032,
  'roe_ttm': 0.283},
 {'symbol': '000001.SZ',
  'security_id': 1000002,
  'trade_date': '2024-01-02',
  'momentum_20d': -0.011,
  'roe_ttm': 0.104}]

## 3. Run the adjusted reference arithmetic

The script version lives in `examples/factor_api_arithmetic_demo.py`. It requests `adjust="forward"` and labels the next-session values as references, not executable prices.

In [4]:
result = run_demo()
print(format_report(result))

QData factor API arithmetic demo
universe=hs300 factor=momentum_20d signal_date=2024-01-02 reference_date=2024-01-03
signal_timing=after_close reference_timing=next_session_forward_adjusted_open_to_close
signal_universe_symbols=2 highest_factor=600519.SH lowest_factor=000001.SZ
highest_factor_marked_change=0.5307% universe_mean_marked_change=0.7883% highest_minus_universe_marked_change=-0.2577% highest_minus_lowest_marked_change=-0.5154%
next_session_tradability_verified=false adjusted_reference_only=true


## 4. What this demonstrates

- The synthetic universe is screened on the signal date only.
- This factor fixture uses latest mode and is filtered to the stated economic date; that date is not a knowledge cutoff, and this mock does not prove SQL version admission.
- The price request uses forward adjustment.
- `adjusted_open_reference`, `adjusted_close_mark`, and `marked_change` make the reference-only semantics explicit.

Next-session tradability is not verified. The fixture checks adjusted reference arithmetic and API/time alignment only. This is not an execution or backtest, executable-price claim, real-market evidence, or investment advice. Real database behavior requires separate integration verification.